# 笔者运行环境
OS: Ubuntu 22.04.1 LTS

Python: 3.12.9

PyTorch: 2.6.0+cu124

In [ ]:
%pip install ipywidgets


In [ ]:
import logging
# 配置日志
logging.basicConfig(
    level=logging.DEBUG,
    format='[%(levelname)s] %(asctime)s %(funcName)s:%(lineno)d :: %(message)s',
    handlers=[
        logging.StreamHandler(),  # 控制台输出
        #logging.FileHandler('notebook.log')  # 文件输出
    ]
)
logger = logging.getLogger("JupyterDemo")


# 安装kagglehub

In [1]:
%pip install kagglehub

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


# 数据集处理

## 下载数据集
找到kaggle上有一个其他人分享的CNN识别骨科影像（X光片）的项目，https://www.kaggle.com/code/ahmedashrafahmed/bone-fracture-classification-using-cnn
不过我们只用他的数据集

In [3]:
import kagglehub
# 默认下载到  ~/.cache/kagglehub/datasets/ 目录中
ahmedashrafahmed_bone_fracture_path = kagglehub.dataset_download('ahmedashrafahmed/bone-fracture')

print(f'Data source import complete. {ahmedashrafahmed_bone_fracture_path}')


[DEBUG] 2025-05-02 09:26:20,729 :: Starting new HTTPS connection (1): www.kaggle.com:443
[DEBUG] 2025-05-02 09:26:21,565 :: https://www.kaggle.com:443 "GET /api/v1/datasets/view/ahmedashrafahmed/bone-fracture HTTP/1.1" 200 None


Data source import complete. /home/guodong/.cache/kagglehub/datasets/ahmedashrafahmed/bone-fracture/versions/1


## 自定义Dataset类
即数据集解析处理类

In [7]:
import os
#import pandas as pd
from torch.utils.data import Dataset
from torchvision.io import read_image
import os

class CustomImageDataset(Dataset):
    def __init__(self, img_dir, transform=None, target_transform=None):
        logger.info(f"img_dir: {img_dir}")
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform
        self.flat_img = []
        for root, dirs, files in os.walk(self.img_dir):
            for dir in dirs:
                dir_path = os.path.join(root, dir)
                for root_, dirs_, files_ in os.walk(dir_path):
                    logger.info(f"label:{dir} image count: {len(files_)}")
                    for file in files_:
                        if file.endswith(".jpg"):
                            self.flat_img.append([os.path.join(root_, file), dir])
        logger.info(f"flat_img[:5]: {self.flat_img[:5]}")

    def __len__(self):
        """
        数据集的样本数量
        """
        return len(self.flat_img)

    def __getitem__(self, idx):
        # logger.info(f"idx: {idx}")
        img_path, label = self.flat_img[idx]
        image = read_image(img_path)
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

## 定义转换函数
transfrom函数用来归一化原始图像数据；target_transform用来归一化原始标签数据

In [49]:
from torchvision.transforms import v2
from torchvision import transforms
from torchvision.io import read_image

transform = v2.Compose([
    v2.Resize((224, 224)),  # 强制统一尺寸
    transforms.Grayscale(num_output_channels=1),  # 强制转单通道
    v2.ToTensor()
])



class CustomTransform:
    def __call__(self, x):
        transform = v2.Compose([
            v2.Resize((224, 224)),  # 强制统一尺寸
            transforms.Grayscale(num_output_channels=1),  # 强制转单通道
            v2.ToTensor()
        ])
        x = transform(x)
        # x dtype是 uint8，转成float并归一化到 [0.0, 1.0] 区间
        x = x.float() / 255.0
        return transform(x)

class CustomTargetTransform:
    def __call__(self, y):
        if y == 'fractured':
            return torch.tensor([1.0])  # dtype为float
        else:
            return torch.tensor([0.0])

transform = CustomTransform()
target_transform = CustomTargetTransform()

/home/guodong/miniconda3/lib/python3.12/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [50]:
train_data = CustomImageDataset(ahmedashrafahmed_bone_fracture_path + "/dataset/train", transform, target_transform)
test_data = CustomImageDataset(ahmedashrafahmed_bone_fracture_path + "/dataset/test", transform, target_transform)
val_data = CustomImageDataset(ahmedashrafahmed_bone_fracture_path + "/dataset/val", transform, target_transform)
print(f"train data: {len(train_data)}, test data: {len(test_data)}, val data: {len(val_data)}")

train data: 4097, test data: 399, val data: 404


## 从Dataset构造DataLoader

In [51]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=64, shuffle=True)

# 模型
定义一个神经网络模型类，用于图像分类任务

In [70]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, class_num=1):
        super(MyModel, self).__init__()
        
        # 卷积块1 (对应Keras第一层)
        self.conv_block1 = nn.Sequential(
            # nn.Conv2d(3, 32, kernel_size=3, stride=1, padding='same'),  # 输入通道3，输出32，padding保持空间维度
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding='same'),  # 输入通道1，输出32，padding保持空间维度
            nn.BatchNorm2d(32),  # 批归一化层，特征数需匹配卷积输出通道
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 池化窗口2x2
        )
        
        # 卷积块2
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.3)
        )
        
        # 卷积块3
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding='same'),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.3)
        )
        
        # 全连接部分
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),  # 输入维度需根据前层输出计算
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, class_num),
            nn.Sigmoid()
        )

    def forward(self, x):
        logger.info(f"{x.dtype}")
        x = self.conv_block1(x)
        logger.info(f"{x.dtype}")
        x = self.conv_block2(x)
        logger.info(f"{x.dtype}")
        x = self.conv_block3(x)
        logger.info(f"{x.dtype}")
        x = self.classifier(x)
        logger.info(f"{x.dtype}")
        return x


# 训练

## 构造模型对象

In [69]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device") 

model = MyModel().to(device)

print(f"model = {model}")


Using cuda device


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 确定损失函数和优化器

In [64]:
# loss_fn = nn.CrossEntropyLoss()  # 多分类任务
loss_fn = nn.BCELoss()  # 二分类任务
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 训练函数

## 关闭info日志
在调试OK，正式开始训练的时候可以关闭info日志，减少输出和训练时间

In [44]:
logger.setLevel(logging.ERROR)

## 评估函数

In [60]:
def evaluate(model, dataloader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            logger.info(f"{pred.shape}, {y.shape}")
            total_loss += loss_fn(pred, y).item()
            print(f"pred: {pred}")
            correct += ((pred > 0.5).long() == y).type(torch.float).sum().item()
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / len(dataloader.dataset)
    return avg_loss, accuracy  # 返回损失和准确率


## 训练函数

In [61]:
import matplotlib.pyplot as plt

def train(dataloader, model, loss_fn, optimizer, epochs):
    best_acc = 0.0  # 用来跟踪最佳准确率
    best_lost = 1.0 # 用来跟踪最佳损失
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(dataloader))

        # 验证阶段
        model.eval()
        val_loss = evaluate(model, val_dataloader, loss_fn)
        val_loss, val_acc = evaluate(model, val_dataloader, loss_fn)
        print(f"Epoch {epoch} Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4%}")
        if val_acc > best_acc:
            best_acc = val_acc
            print(f"[Save Model] Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4%}")
            torch.save(model.state_dict(), 'best_model.pth')
        elif val_acc == best_acc and val_loss < best_lost:
            best_lost = val_loss
            print(f"[Save Model] Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4%}")
            torch.save(model.state_dict(), 'best_model.pth')
        val_losses.append(val_loss)


    # 绘制损失曲线
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.legend()
    plt.show()

train(train_dataloader, model, loss_fn, optimizer, 20)

/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [0,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [1,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [2,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [3,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [4,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [5,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [6,0,0] Assertion `inpu

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# 加载训练好的模型

In [101]:
classes = [
    "未骨折",
    "骨折",
]

In [114]:
best_model = MyModel().to(device)
best_model.load_state_dict(torch.load('best_model.pth', weights_only=True))
best_model.eval()
x, y = test_data[0][0], test_data[0][1]
print(x, y)
with torch.no_grad():
    x = x.unsqueeze(0).to(device)
    pred = best_model(x)
    pred = pred.unsqueeze(1)
    predicted = classes[(pred > 0.5).long().item()]
    print(f"y type:{y.type()}")
    actual = classes[y.long()]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')


image dtype: torch.uint8
new image dtype: torch.float32
image dtype: torch.uint8
new image dtype: torch.float32
tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]]) tensor(1.)
y type:torch.FloatTensor
Predicted: "骨折", Actual: "骨折"
